# Cross-dataset DPE study, step 1: SubpopBench ERM backbones and frozen features

Plan: `experiments/dpe_cross_dataset/PLAN.md`. Select **GPU T4 x2**, **Internet on**, then
**Save Version -> Save & Run All**. One session trains the two backbone seeds in `SEEDS` (one per GPU)
for `DATASET` and saves validation/test features for the CPU notebook (`nested_kaggle.ipynb`).

* **CelebA:** first **Add Input -> Datasets -> `jessicali9530/celeba-dataset`** (images and attribute
  CSVs). Metadata is generated by SubpopBench's own `generate_metadata_celeba`.
* **CivilCommentsFine:** downloads the WILDS bundle SubpopBench uses, or attach a dataset containing
  `all_data_with_identities.csv`.
* **MultiNLI:** only if the team accepted PLAN Sect. 9. Downloads SubpopBench's cached BERT features.

The notebook first runs a timing smoke test (300 steps) and a 30-step end-to-end test with feature
extraction and a resume check. If two 30,001-step backbones would not finish before `TIME_LIMIT_HOURS`,
it stops before the main run unless `ALLOW_RESUME = True` (then the run is cut at the limit and the next
session continues it from this notebook's output; resumed backbones are flagged as a deviation).

Seeds per dataset: 0-4 (PLAN Sect. 3). Run seeds [0, 1], [2, 3] and [4] in separate sessions/accounts.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, shutil, glob, hashlib, signal
T0 = time.time()
DATASET = 'CelebA'          # 'CelebA' | 'CivilCommentsFine' | 'MultiNLI'
SEEDS = [0, 1]              # one per GPU; at most two
RUN_SMOKE = True
TIMING_STEPS = 300
TIME_LIMIT_HOURS = 11.3     # Kaggle stops at 12 h; leave time to save
ALLOW_RESUME = False        # True: allow cutting a run at the limit and resuming it next session
WORKERS_PER_JOB = 2
FULL_STEPS = 30001          # SubpopBench default for all three datasets (checked below)

WORK = Path('/kaggle/working/xdpe-backbones')
CODE = WORK / 'code'
RUNS = WORK / 'runs'
SMOKE = WORK / 'smoke'
for p in (CODE, RUNS):
    p.mkdir(parents=True, exist_ok=True)
assert len(SEEDS) <= 2 and DATASET in ('CelebA', 'CivilCommentsFine', 'MultiNLI')

def hours():
    return (time.time() - T0) / 3600
print(subprocess.check_output(['nvidia-smi'], text=True))

## Pinned environment

In [ ]:
# Same pinned environment as the Waterbirds fresh-backbone runs (noerula/dpe-corrected-label), plus gdown.
BOOT = WORK / 'bootstrap'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', '--target', str(BOOT), 'uv==0.9.8'], check=True)
UV_ENV = dict(os.environ, PYTHONPATH=str(BOOT), UV_CACHE_DIR=str(WORK / 'uv-cache'), UV_PYTHON_INSTALL_DIR=str(WORK / 'python'))
VENV = WORK / 'venv'
PY = VENV / 'bin/python'
if not PY.exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.11', '--seed', str(VENV)], env=UV_ENV, check=True)
ISOLATED_ENV = dict(os.environ, PYTHONNOUSERSITE='1')
for key in ('PYTHONHOME', 'PYTHONPATH'):
    ISOLATED_ENV.pop(key, None)
subprocess.run([str(PY), '-m', 'pip', 'install', '--quiet', 'torch==2.6.0+cu118', 'torchvision==0.21.0+cu118',
                '--index-url', 'https://download.pytorch.org/whl/cu118'], env=ISOLATED_ENV, check=True)
PINS = ['Pillow==10.3.0', 'matplotlib==3.8.3', 'numpy==1.26.4', 'pandas==2.2.3', 'scikit-learn==1.3.2', 'scipy==1.12.0',
        'tqdm==4.66.2', 'transformers==4.46.3', 'timm==1.0.12', 'tensorboard==2.18.0', 'tensorboard-logger==0.1.0',
        'protobuf==3.20.3', 'gdown==5.2.0']
subprocess.run([str(PY), '-m', 'pip', 'install', '--quiet', *PINS], env=ISOLATED_ENV, check=True)
subprocess.run([str(PY), '-m', 'pip', 'check'], env=ISOLATED_ENV, check=True)
# SubpopBench calls torch.load without weights_only on its own checkpoints (resume) and on the MultiNLI
# cached features; torch 2.6 defaults to weights_only=True. These files are trusted (SubpopBench's own
# outputs and the GroupDRO feature bundle), so restore the old default for the training processes.
BASE_ENV = dict(ISOLATED_ENV, TORCH_HOME=str(WORK / 'cache/torch'), HF_HOME=str(WORK / 'cache/hf'),
                MPLCONFIGDIR=str(WORK / 'cache/matplotlib'), MPLBACKEND='Agg', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2',
                TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1', TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD='1')
subprocess.run([str(PY), '-c', "import torch; assert torch.cuda.device_count() == 2, 'Choose Kaggle T4 x2'; "
                "print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(2)])"], env=BASE_ENV, check=True)

## Pinned SubpopBench and project code

In [ ]:
def checkout(url, destination, commit):
    if not destination.exists():
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    assert subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip() == commit
    assert not subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain', '--untracked-files=no'], text=True)

SUBPOP = WORK / 'subpopbench-upstream'
checkout('https://github.com/YyzHarry/SubpopBench.git', SUBPOP, '4d3dbbe21029666ef19d040e110ec22908640c5b')
steps = subprocess.check_output([str(PY), '-c', f"from subpopbench.dataset import datasets as D; print(D.{DATASET}.N_STEPS)"],
                                cwd=SUBPOP, env=BASE_ENV, text=True).strip()
assert int(steps) == FULL_STEPS, steps

In [ ]:
%%writefile "{CODE}/prepare_data.py"
#!/usr/bin/env python3
"""Lay out one dataset the way SubpopBench expects and generate its metadata with SubpopBench's own code.

    python prepare_data.py --subpop-root SUBPOP --data-dir DATA --dataset CelebA [--search /kaggle/input]

Metadata always comes from `subpopbench/scripts/download.py` (generate_metadata_*) at the pinned
commit, so labels, attributes, splits and the CivilComments "fine" groups are SubpopBench's.
Raw files are taken from attached inputs when found under --search, otherwise downloaded from the
URLs SubpopBench itself uses.

CelebA: the Kaggle dataset `jessicali9530/celeba-dataset` ships CSV versions of the official
list_attr_celeba.txt / list_eval_partition.txt; they are rewritten in the official text layout
(same rows, same order, values -1/1) before SubpopBench's generator reads them.
"""
import argparse
import csv
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path


def find(roots, name):
    hits = []
    for root in roots:
        if Path(root).exists():
            hits += [p for p in Path(root).rglob(name)]
    return sorted(hits)


def safe_extract(archive, target):
    with tarfile.open(archive) as tar:
        for m in tar.getmembers():
            dest = (Path(target) / m.name).resolve()
            if not dest.is_relative_to(Path(target).resolve()) or m.issym() or m.islnk() or m.isdev():
                raise RuntimeError(f"Unexpected archive entry: {m.name}")
        tar.extractall(target)


def download(url, target):
    part = Path(str(target) + ".part")
    urllib.request.urlretrieve(url, part)
    part.replace(target)


def generator(subpop_root, name):
    sys.path.insert(0, str(subpop_root))
    from subpopbench.scripts import download as d
    return getattr(d, name)


def celeba(a):
    root = a.data_dir / "celeba"
    root.mkdir(parents=True, exist_ok=True)
    images = root / "img_align_celeba"
    if not images.exists():
        firsts = [p.parent for p in find(a.search, "000001.jpg")]
        if not firsts:
            raise SystemExit("CelebA images not found: attach the Kaggle dataset jessicali9530/celeba-dataset")
        images.symlink_to(firsts[0].resolve(), target_is_directory=True)
    attr_txt, part_txt = root / "list_attr_celeba.txt", root / "list_eval_partition.txt"
    if not attr_txt.exists():
        txt = find(a.search, "list_attr_celeba.txt")
        if txt:
            shutil.copy(txt[0], attr_txt)
        else:
            src = find(a.search, "list_attr_celeba.csv")
            if not src:
                raise SystemExit("list_attr_celeba not found")
            with src[0].open(newline="") as f:
                rows = list(csv.reader(f))
            header, body = rows[0], rows[1:]
            with attr_txt.open("w") as f:
                f.write(f"{len(body)}\n")
                f.write(" ".join(header[1:]) + "\n")
                for r in body:
                    f.write(" ".join(r) + "\n")
    if not part_txt.exists():
        txt = find(a.search, "list_eval_partition.txt")
        if txt:
            shutil.copy(txt[0], part_txt)
        else:
            src = find(a.search, "list_eval_partition.csv")
            if not src:
                raise SystemExit("list_eval_partition not found")
            with src[0].open(newline="") as f:
                rows = list(csv.reader(f))[1:]
            with part_txt.open("w") as f:
                for r in rows:
                    f.write(f"{r[0]} {r[1]}\n")
    # Header check: Blond_Hair is attribute 10 and Male attribute 21 in the official order.
    lines = attr_txt.read_text().splitlines()
    names = lines[1].split()
    if names[9] != "Blond_Hair" or names[20] != "Male":
        raise SystemExit(f"Unexpected attribute order: {names[:21]}")
    # SubpopBench's generator pairs the two files line by line; the image ids must agree.
    attr_ids = [l.split()[0] for l in lines[2:]]
    part_ids = [l.split()[0] for l in part_txt.read_text().splitlines() if l.strip()]
    if attr_ids != part_ids or len(attr_ids) != 202599:
        raise SystemExit("list_attr_celeba and list_eval_partition disagree on image order or count")
    generator(a.subpop_root, "generate_metadata_celeba")(str(a.data_dir))


def civilcomments(a):
    root = a.data_dir / "civilcomments"
    root.mkdir(parents=True, exist_ok=True)
    target = root / "all_data_with_identities.csv"
    if not target.exists():
        hits = find(a.search, "all_data_with_identities.csv")
        if hits:
            target.symlink_to(hits[0].resolve())
        else:
            archive = root / "civilcomments.tar.gz"
            # The WILDS CivilComments v1.0 bundle used by SubpopBench's download_civilcomments.
            download("https://worksheets.codalab.org/rest/bundles/0x8cd3de0634154aeaad2ee6eb96723c6e/contents/blob/", archive)
            safe_extract(archive, root)
            archive.unlink()
            if not target.exists():
                found = list(root.rglob("all_data_with_identities.csv"))
                if not found:
                    raise SystemExit("all_data_with_identities.csv missing from the archive")
                target.symlink_to(found[0].resolve())
    generator(a.subpop_root, "generate_metadata_civilcomments")(str(a.data_dir))


def multinli(a):
    root = a.data_dir / "multinli"
    glue = root / "glue_data/MNLI"
    glue.mkdir(parents=True, exist_ok=True)
    (root / "data").mkdir(parents=True, exist_ok=True)
    needed = ["cached_train_bert-base-uncased_128_mnli", "cached_dev_bert-base-uncased_128_mnli",
              "cached_dev_bert-base-uncased_128_mnli-mm"]
    if not all((glue / n).exists() for n in needed):
        hits = {n: find(a.search, n) for n in needed}
        if all(hits.values()):
            for n in needed:
                (glue / n).symlink_to(hits[n][0].resolve())
        else:
            archive = glue / "multinli_bert_features.tar.gz"
            download("https://nlp.stanford.edu/data/dro/multinli_bert_features.tar.gz", archive)
            safe_extract(archive, glue)
            archive.unlink()
    meta = root / "data/metadata_random.csv"
    if not meta.exists():
        hits = find(a.search, "metadata_random.csv")
        if hits:
            shutil.copy(hits[0], meta)
        else:
            download("https://raw.githubusercontent.com/kohpangwei/group_DRO/master/dataset_metadata/multinli/metadata_random.csv", meta)
    generator(a.subpop_root, "generate_metadata_multinli")(str(a.data_dir))


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--subpop-root", type=Path, required=True)
    p.add_argument("--data-dir", type=Path, required=True)
    p.add_argument("--dataset", choices=["CelebA", "CivilCommentsFine", "MultiNLI"], required=True)
    p.add_argument("--search", nargs="*", default=["/kaggle/input"])
    a = p.parse_args()
    a.data_dir = a.data_dir.resolve()
    {"CelebA": celeba, "CivilCommentsFine": civilcomments, "MultiNLI": multinli}[a.dataset](a)
    print("prepared", a.dataset, "in", a.data_dir)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile "{CODE}/train_backbone.py"
#!/usr/bin/env python3
"""Train one SubpopBench ERM backbone and save frozen validation/test features (PLAN.md Sect. 3).

Runs inside the pinned environment on exactly one visible GPU:

    python train_backbone.py --subpop-root SUBPOP --data-dir DATA --dataset CelebA --seed 0 --output-dir OUT
        [--steps 300]            timing smoke only; the real run keeps SubpopBench's default (30,001)
        [--resume MODEL.pkl]     continue an interrupted run (recorded as a deviation, see below)

Training: `subpopbench.train --algorithm ERM --train_attr no --hparams_seed 0` at the pinned
commit, default hyper-parameters and steps, unmodified. The backbone is SubpopBench's last
checkpoint (`model.pkl`, written at the final step), as for the Waterbirds fresh backbones.

Features: the ERM featurizer output (ResNet-50: 2,048-d pooled; BERT: 768-d pooler output, with
dropout off in eval mode) on the SubpopBench validation and test datasets, same transforms and
tokenisation as SubpopBench evaluation. Saved as features/{val,test}.npz with ids, y, a and a
`cluster` key (CivilComments: SHA-1 of the comment text, since one comment appears once per
identity it mentions; otherwise the id). The ERM classifier's own validation/test metrics are
stored in backbone.json for context.

Resume: SubpopBench's --resume restores weights and step count, but not optimizer state, the
text learning-rate schedule or the data order. A resumed backbone is therefore NOT the same as an
uninterrupted one; backbone.json records `resumed_from_step` so the deviation is reported.
"""
import argparse
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path

SUBPOP_COMMIT = "4d3dbbe21029666ef19d040e110ec22908640c5b"
FEATURE_DIM = {"CelebA": 2048, "CivilCommentsFine": 768, "MultiNLI": 768}
# Expected (split -> group counts in (y, a) order 00, 01, 10, 11) for CelebA, from GroupDRO/SubpopBench.
# A mismatch is recorded and printed, not fatal (PLAN Sect. 2: verified counts are used).
EXPECTED = {"CelebA": {1: [8535, 8276, 2874, 182], 2: [9767, 7535, 2480, 180]}}
METADATA = {"CelebA": "celeba/metadata_celeba.csv", "MultiNLI": "multinli/metadata_multinli.csv",
            "CivilCommentsFine": "civilcomments/metadata_civilcomments_fine.csv"}


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=1, sort_keys=True, default=str) + "\n")
    tmp.replace(path)


def verify_repo(path):
    actual = subprocess.check_output(["git", "-C", str(path), "rev-parse", "HEAD"], text=True).strip()
    if actual != SUBPOP_COMMIT:
        raise RuntimeError(f"Expected SubpopBench {SUBPOP_COMMIT}, found {actual}")
    if subprocess.check_output(["git", "-C", str(path), "status", "--porcelain", "--untracked-files=no"], text=True):
        raise RuntimeError("SubpopBench tracked files have been modified")


def verify_data(data_dir, dataset):
    import pandas as pd
    meta = Path(data_dir) / METADATA[dataset]
    df = pd.read_csv(meta)
    counts = {}
    for split in (0, 1, 2):
        part = df[df.split == split]
        counts[split] = {f"{y}{a}": int(n) for (y, a), n in part.groupby(["y", "a"]).size().items()}
    out = dict(metadata=str(meta), metadata_sha256=sha256(meta), rows=len(df), group_counts=counts)
    if dataset in EXPECTED:
        observed = {s: [counts[s].get(g, 0) for g in ("00", "01", "10", "11")] for s in EXPECTED[dataset]}
        out["expected_group_counts"] = EXPECTED[dataset]
        out["matches_expected"] = observed == EXPECTED[dataset]
        if not out["matches_expected"]:
            print(f"WARNING: {dataset} group counts {observed} differ from expected {EXPECTED[dataset]}", flush=True)
    return out


def train(args, out):
    bootstrap = ("import runpy; from subpopbench.dataset import datasets as D; "
                 f"D.{args.dataset}.N_WORKERS = {args.workers}; "
                 "runpy.run_module('subpopbench.train', run_name='__main__')")
    cmd = [sys.executable, "-c", bootstrap, "--algorithm", "ERM", "--dataset", args.dataset, "--train_attr", "no",
           "--data_dir", str(args.data_dir), "--output_dir", str(out / "subpopbench"),
           "--output_folder_name", "standard", "--seed", str(args.seed), "--hparams_seed", "0"]
    if args.steps:
        cmd += ["--steps", str(args.steps)]
    if args.resume:
        cmd += ["--resume", str(args.resume)]
    write_json(out / "train_command.json", dict(argv=cmd, cwd=str(args.subpop_root)))
    started = time.time()
    with open(out / "train.log", "a") as log:
        subprocess.run(cmd, cwd=args.subpop_root, stdout=log, stderr=subprocess.STDOUT, check=True)
    run_dir = out / "subpopbench/standard_attrNo" / f"{args.dataset}_ERM_hparams0_seed{args.seed}"
    if not (run_dir / "done").exists():
        raise RuntimeError("SubpopBench did not finish")
    return run_dir, time.time() - started


def timing(run_dir, wall_seconds, steps_requested):
    """Per-step time from SubpopBench's results.json; evaluation time is the remainder."""
    lines = [json.loads(l) for l in (run_dir / "results.json").read_text().splitlines() if l.strip()]
    steps = lines[-1]["step"] + 1
    # Each line averages step_time over the steps since the previous checkpoint.
    spans, prev = [], -1
    for line in lines:
        spans.append((line["step"] - prev, line["step_time"]))
        prev = line["step"]
    step_seconds = sum(n * t for n, t in spans) / sum(n for n, _ in spans)
    evals = len(lines) + 1  # every checkpoint plus the final evaluation
    eval_seconds = max(0., wall_seconds - steps * step_seconds) / evals
    return dict(steps=steps, seconds_per_step=step_seconds, evaluations=evals,
                seconds_per_evaluation=eval_seconds, wall_seconds=wall_seconds)


def extract(args, run_dir, out, device):
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    sys.path.insert(0, str(args.subpop_root))
    from subpopbench.dataset import datasets
    from subpopbench.learning import algorithms
    ckpt = torch.load(run_dir / "model.pkl", map_location="cpu", weights_only=False)  # our own SubpopBench output
    hparams = ckpt["model_hparams"]
    cls = datasets.get_dataset_class(args.dataset)
    splits = {name: cls(str(args.data_dir), split, hparams) for name, split in (("val", "va"), ("test", "te"))}
    erm = algorithms.ERM(splits["val"].data_type, ckpt["model_input_shape"], ckpt["num_labels"],
                         ckpt["num_attributes"], 0, hparams)
    erm.load_state_dict(ckpt["model_dict"], strict=True)
    erm.to(device).eval()
    results, metrics = {}, {}
    for name, ds in splits.items():
        loader = DataLoader(ds, batch_size=256 if ds.data_type == "images" else 128, shuffle=False,
                            num_workers=args.workers)
        feats, logits, index = [], [], []
        with torch.no_grad():
            for i, x, y, a in loader:
                z = erm.featurizer(x.to(device))
                feats.append(z.float().cpu().numpy())
                logits.append(erm.classifier(z).float().cpu().numpy())
                index.append(i.numpy())
        feats, logits, index = np.concatenate(feats), np.concatenate(logits), np.concatenate(index)
        if not np.array_equal(index, np.arange(len(ds))):
            raise RuntimeError("Unexpected loader order")
        if feats.shape != (len(ds), FEATURE_DIM[args.dataset]) or not np.isfinite(feats).all() or (feats.std(1) == 0).any():
            raise ValueError("Invalid features")
        y = np.asarray(ds.y, dtype=int)
        a = np.asarray(ds.a, dtype=int)
        ids = np.array([os.path.basename(p) if ds.data_type == "images" else str(p) for p in ds.x])
        if len(set(ids)) != len(ids):
            raise ValueError("Duplicate ids")
        if args.dataset.startswith("CivilComments"):
            cluster = np.array([hashlib.sha1(ds.text_array[int(p)].encode("utf-8")).hexdigest() for p in ds.x])
        else:
            cluster = ids
        target = out / "features" / f"{name}.npz"
        target.parent.mkdir(parents=True, exist_ok=True)
        np.savez(target, ids=ids, features=feats.astype(np.float32), y=y, a=a, cluster=cluster)
        results[name] = target
        pred = logits.argmax(1)
        groups = np.array([f"{u}{v}" for u, v in zip(y, a)])
        per_group = {g: dict(n=int((groups == g).sum()), accuracy=float((pred == y)[groups == g].mean()))
                     for g in sorted(set(groups))}
        metrics[name] = dict(overall=float((pred == y).mean()), wga=min(v["accuracy"] for v in per_group.values()),
                             per_group=per_group, duplicated_rows=int(len(cluster) - len(set(cluster))))
    return results, metrics


def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--subpop-root", type=Path, required=True)
    p.add_argument("--data-dir", type=Path, required=True)
    p.add_argument("--dataset", choices=sorted(FEATURE_DIM), required=True)
    p.add_argument("--seed", type=int, required=True)
    p.add_argument("--output-dir", type=Path, required=True)
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--steps", type=int, default=0)
    p.add_argument("--resume", type=Path)
    p.add_argument("--skip-features", action="store_true", help="timing smoke: train only")
    args = p.parse_args()
    for name in ("subpop_root", "data_dir", "output_dir"):
        setattr(args, name, getattr(args, name).resolve())
    verify_repo(args.subpop_root)
    args.output_dir.mkdir(parents=True, exist_ok=True)
    import torch
    import torchvision
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError("Launch with CUDA_VISIBLE_DEVICES=<one GPU>")
    manifest_path = args.output_dir / "backbone.json"
    manifest = dict(dataset=args.dataset, seed=args.seed, status="running", smoke_steps=args.steps or None,
                    subpopbench=SUBPOP_COMMIT, runner_sha256=sha256(__file__),
                    environment=dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,
                                     cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0)),
                    data=verify_data(args.data_dir, args.dataset))
    if args.resume:
        resumed = torch.load(args.resume, map_location="cpu", weights_only=False)["start_step"]
        manifest["resumed_from_step"] = int(resumed)
        manifest["deviation"] = ("resumed with SubpopBench --resume: optimizer state, text LR schedule and "
                                 "data order restart; not identical to an uninterrupted run")
    write_json(manifest_path, manifest)
    try:
        run_dir, wall = train(args, args.output_dir)
        manifest["timing"] = timing(run_dir, wall, args.steps)
        manifest["checkpoint"] = str(run_dir / "model.pkl")
        manifest["checkpoint_sha256"] = sha256(run_dir / "model.pkl")
        write_json(manifest_path, manifest)
        if not args.skip_features:
            files, metrics = extract(args, run_dir, args.output_dir, "cuda")
            manifest["features_sha256"] = {k: sha256(v) for k, v in files.items()}
            manifest["erm_metrics"] = metrics
        manifest["status"] = "complete" if not args.steps else "smoke-complete"
    except BaseException as e:
        manifest["status"] = "failed"
        manifest["error"] = repr(e)
        raise
    finally:
        write_json(manifest_path, manifest)
    print(json.dumps({k: manifest.get(k) for k in ("status", "timing", "erm_metrics")}, default=str)[:2000], flush=True)


if __name__ == "__main__":
    main()

## Data (SubpopBench metadata generators)

In [ ]:
DATA = WORK / 'data'
subprocess.run([str(PY), str(CODE / 'prepare_data.py'), '--subpop-root', str(SUBPOP), '--data-dir', str(DATA),
                '--dataset', DATASET, '--search', '/kaggle/input'], cwd=SUBPOP, env=BASE_ENV, check=True)

## Jobs, timing smoke test and resume check

In [ ]:
def launch(name, seed, gpu, out, extra=()):
    cmd = [str(PY), str(CODE / 'train_backbone.py'), '--subpop-root', str(SUBPOP), '--data-dir', str(DATA),
           '--dataset', DATASET, '--seed', str(seed), '--output-dir', str(out), '--workers', str(WORKERS_PER_JOB), *extra]
    out.mkdir(parents=True, exist_ok=True)
    log = open(out / 'runner.log', 'a')
    env = dict(BASE_ENV, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONHASHSEED=str(seed))
    proc = subprocess.Popen(cmd, cwd=SUBPOP, env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    print(f'[{hours():.2f} h] started {name} on GPU {gpu}', flush=True)
    return dict(name=name, proc=proc, log=log, out=out, start=time.time())

def stop(run):
    if run['proc'].poll() is None:
        os.killpg(run['proc'].pid, signal.SIGTERM)
        try:
            run['proc'].wait(timeout=60)
        except subprocess.TimeoutExpired:
            os.killpg(run['proc'].pid, signal.SIGKILL); run['proc'].wait()

def wait(runs, hard_limit_hours=None):
    while any(r['proc'].poll() is None for r in runs):
        if hard_limit_hours and hours() > hard_limit_hours:
            print('Time limit reached: stopping running jobs (last SubpopBench checkpoint is kept).', flush=True)
            for r in runs:
                stop(r)
            break
        time.sleep(30)
    status = {}
    for r in runs:
        r['log'].close()
        manifest = json.loads((r['out'] / 'backbone.json').read_text()) if (r['out'] / 'backbone.json').exists() else {}
        status[r['name']] = manifest.get('status', 'no manifest')
        print(f"[{hours():.2f} h] {r['name']}: {status[r['name']]} after {(time.time() - r['start']) / 3600:.2f} h", flush=True)
        if r['proc'].returncode not in (0, None) and status[r['name']] != 'complete':
            print((r['out'] / 'runner.log').read_text(errors='replace')[-3000:])
    return status

In [ ]:
ESTIMATE_HOURS = None
if RUN_SMOKE:
    shutil.rmtree(SMOKE, ignore_errors=True)
    timing_run = launch('timing', 99, 0, SMOKE / 'timing', ['--steps', str(TIMING_STEPS), '--skip-features'])
    e2e_run = launch('end-to-end', 98, 1, SMOKE / 'e2e', ['--steps', '30'])
    status = wait([timing_run, e2e_run])
    assert status == {'timing': 'smoke-complete', 'end-to-end': 'smoke-complete'}, status
    # Resume mechanism: continue the 30-step run to 40 steps from its model.pkl.
    ckpt = next((SMOKE / 'e2e').rglob('model.pkl'))
    shutil.copy(ckpt, SMOKE / 'resume_from.pkl')
    status = wait([launch('resume', 98, 1, SMOKE / 'e2e', ['--steps', '40', '--skip-features',
                                                         '--resume', str(SMOKE / 'resume_from.pkl')])])
    log_text = '\n'.join(p.read_text(errors='replace') for p in (SMOKE / 'e2e').rglob('out.txt'))
    resume_ok = status['resume'] == 'smoke-complete' and "Loaded checkpoint" in log_text
    t = json.loads((SMOKE / 'timing/backbone.json').read_text())['timing']
    e2e = json.loads((SMOKE / 'e2e/backbone.json').read_text())
    evaluations = FULL_STEPS // 1000 + 2
    ESTIMATE_HOURS = (FULL_STEPS * t['seconds_per_step'] + evaluations * t['seconds_per_evaluation']) / 3600 + 0.3
    report = dict(dataset=DATASET, timing=t, estimate_hours_per_backbone=ESTIMATE_HOURS, resume_mechanism_ok=resume_ok,
                  data=e2e['data'], erm_smoke_metrics=e2e.get('erm_metrics', {}).get('val'))
    (WORK / 'smoke_report.json').write_text(json.dumps(report, indent=1, default=str))
    print(json.dumps(report, indent=1, default=str)[:3000])
    print(f'Smoke passed after {hours():.2f} h. Estimated {ESTIMATE_HOURS:.1f} h per backbone (two run in parallel).')

## Main run: one backbone per GPU

In [ ]:
# Continue interrupted runs of the same dataset/seed from an attached earlier output (ALLOW_RESUME only).
RESUME = {}
for manifest in sorted(glob.glob('/kaggle/input/**/backbone.json', recursive=True)):
    m = json.loads(Path(manifest).read_text())
    if m.get('dataset') != DATASET or m.get('seed') not in SEEDS or m.get('smoke_steps') or m.get('status') == 'complete':
        continue
    models = sorted(Path(manifest).parent.rglob('model.pkl'))
    if models and ALLOW_RESUME:
        target = RUNS / f"{DATASET}-seed{m['seed']}"
        if not target.exists():
            shutil.copytree(Path(manifest).parent, target, ignore=shutil.ignore_patterns('features'))
        shutil.copy(models[0], target / 'resume_from.pkl')
        RESUME[m['seed']] = target / 'resume_from.pkl'
        print('resuming seed', m['seed'], 'from', models[0])
for manifest in sorted(glob.glob('/kaggle/input/**/backbone.json', recursive=True)):
    m = json.loads(Path(manifest).read_text())
    if m.get('dataset') == DATASET and m.get('seed') in SEEDS and m.get('status') == 'complete' and not m.get('smoke_steps'):
        print(f"seed {m['seed']} is already complete in an attached input ({manifest}); it will be skipped")
        SEEDS = [s for s in SEEDS if s != m['seed']]

In [ ]:
fits = ESTIMATE_HOURS is None or hours() + 1.1 * ESTIMATE_HOURS < TIME_LIMIT_HOURS
if not fits and not ALLOW_RESUME:
    raise SystemExit(f'A {FULL_STEPS}-step backbone needs about {ESTIMATE_HOURS:.1f} h, which does not fit in '
                     f'{TIME_LIMIT_HOURS} h. Set ALLOW_RESUME = True to train across two sessions (recorded as a deviation).')
runs = []
for gpu, seed in enumerate(SEEDS):
    extra = ['--resume', str(RESUME[seed])] if seed in RESUME else []
    runs.append(launch(f'{DATASET}-seed{seed}', seed, gpu, RUNS / f'{DATASET}-seed{seed}', extra))
status = wait(runs, hard_limit_hours=TIME_LIMIT_HOURS)
print(json.dumps(status, indent=1))

## Clean up and summarise

In [ ]:
# Keep: backbone.json manifests, features, SubpopBench logs/results and model.pkl. Drop the environment and data.
for name in ('venv', 'uv-cache', 'python', 'bootstrap', 'cache', 'data', 'subpopbench-upstream'):
    p = WORK / name
    if p.is_symlink() or p.is_file():
        p.unlink()
    elif p.exists():
        shutil.rmtree(p, ignore_errors=True)
for p in (SMOKE.rglob('model.pkl') if SMOKE.exists() else []):
    p.unlink()
summary = {}
for manifest in sorted(RUNS.glob('*/backbone.json')):
    m = json.loads(manifest.read_text())
    summary[manifest.parent.name] = dict(status=m['status'], resumed_from_step=m.get('resumed_from_step'),
                                         hours=round(m.get('timing', {}).get('wall_seconds', 0) / 3600, 2))
(WORK / 'session_summary.json').write_text(json.dumps(summary, indent=1))
print(json.dumps(summary, indent=1))
print(f'Done after {hours():.2f} h. Attach this notebook output to nested_kaggle.ipynb once all seeds are complete.')